# RSM vs ANN: adsorption capacity of 5 biochar materials

**Goals**
1. Optimise Concentration, Time, pH and Dosage for each material.
2. Find the best of the 5 materials using ANOVA.
3. Actual vs predicted adsorption capacity (qe, mg/g) for each material.

**Constraints (from the experimenter)**
- Optimum pH must be **4–8** (pH 12 is not practical for farmers).
- Response is adsorption capacity qe; low dosage is expected to give the highest qe.
- Biochar is the **control**: its negative qe values are genuine lab data and it is not optimised.

**No data leakage**
- Train/test split is by *run condition*: all 3 replicates of a run, and all 5 materials at that condition, stay on the same side.
- Centre runs 1–4 share identical conditions, so they count as one condition.
- Test conditions (6 runs) are drawn at random but stratified: 1 from each factor's one-factor block (runs 5–8, 9–12, 13–16, 17–20) and 2 from the factorial block (21–28). Centre runs and the extreme corner runs 29–32 always stay in training, so test points lie inside the tested range.
- All tuning (RSM term removal, ANN neurons and regularisation, input scaling) uses training data only, via grouped inner cross-validation.
- The test set is scored once. A leave-one-condition-out (LOCO) estimate, with the full fit-and-tune repeated inside every fold, is reported alongside because 6 test points is a small sample.

Coding: A = Concentration (10, 25, 50, 75, 100 ppm), B = Time (60, 120, 180, 240, 360 min), C = pH (4–12), D = Dosage (10, 20, 50, 100, 150 mg) → coded −2 … +2.

In [ ]:
import io, os, zipfile, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm, statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy import stats
from scipy.optimize import minimize
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200, 'display.max_columns', 30)
SEED = 42
OUT = 'results'; os.makedirs(OUT, exist_ok=True)

In [ ]:
# @title Lab data (hidden: 480 measured rows, nothing added)
CSV = """mat,run,conc,time,pH,dose,x1,x2,x3,x4,rep,y
Biochar,1,50,180,8,50,0,0,0,0,R1,-1.960784
Biochar,1,50,180,8,50,0,0,0,0,R2,-2.941176
Biochar,1,50,180,8,50,0,0,0,0,R3,-1.960784
Biochar,2,50,180,8,50,0,0,0,0,R1,-3.186275
Biochar,2,50,180,8,50,0,0,0,0,R2,-1.22549
Biochar,2,50,180,8,50,0,0,0,0,R3,-1.22549
Biochar,3,50,180,8,50,0,0,0,0,R1,-1.470588
Biochar,3,50,180,8,50,0,0,0,0,R2,-1.960784
Biochar,3,50,180,8,50,0,0,0,0,R3,-2.205882
Biochar,4,50,180,8,50,0,0,0,0,R1,-0.980392
Biochar,4,50,180,8,50,0,0,0,0,R2,0.245098
Biochar,4,50,180,8,50,0,0,0,0,R3,-0.735294
Biochar,5,25,180,8,50,-1,0,0,0,R1,-0.245098
Biochar,5,25,180,8,50,-1,0,0,0,R2,-1.22549
Biochar,5,25,180,8,50,-1,0,0,0,R3,-0.735294
Biochar,6,75,180,8,50,1,0,0,0,R1,0.980392
Biochar,6,75,180,8,50,1,0,0,0,R2,2.45098
Biochar,6,75,180,8,50,1,0,0,0,R3,1.715686
Biochar,7,10,180,8,50,-2,0,0,0,R1,-0.784314
Biochar,7,10,180,8,50,-2,0,0,0,R2,-1.642157
Biochar,7,10,180,8,50,-2,0,0,0,R3,-1.151961
Biochar,8,100,180,8,50,2,0,0,0,R1,2.205882
Biochar,8,100,180,8,50,2,0,0,0,R2,3.431373
Biochar,8,100,180,8,50,2,0,0,0,R3,1.715686
Biochar,9,50,120,8,50,0,-1,0,0,R1,-0.245098
Biochar,9,50,120,8,50,0,-1,0,0,R2,-0.245098
Biochar,9,50,120,8,50,0,-1,0,0,R3,-0.245098
Biochar,10,50,240,8,50,0,1,0,0,R1,-1.715686
Biochar,10,50,240,8,50,0,1,0,0,R2,-2.45098
Biochar,10,50,240,8,50,0,1,0,0,R3,-2.205882
Biochar,11,50,60,8,50,0,-2,0,0,R1,-1.470588
Biochar,11,50,60,8,50,0,-2,0,0,R2,-2.205882
Biochar,11,50,60,8,50,0,-2,0,0,R3,-2.205882
Biochar,12,50,360,8,50,0,2,0,0,R1,-2.941176
Biochar,12,50,360,8,50,0,2,0,0,R2,-1.715686
Biochar,12,50,360,8,50,0,2,0,0,R3,-2.696078
Biochar,13,50,180,6,50,0,0,-1,0,R1,-0.245098
Biochar,13,50,180,6,50,0,0,-1,0,R2,-2.696078
Biochar,13,50,180,6,50,0,0,-1,0,R3,-2.205882
Biochar,14,50,180,10,50,0,0,1,0,R1,0.245098
Biochar,14,50,180,10,50,0,0,1,0,R2,-0.490196
Biochar,14,50,180,10,50,0,0,1,0,R3,0.245098
Biochar,15,50,180,4,50,0,0,-2,0,R1,-1.960784
Biochar,15,50,180,4,50,0,0,-2,0,R2,-1.715686
Biochar,15,50,180,4,50,0,0,-2,0,R3,-1.22549
Biochar,16,50,180,12,50,0,0,2,0,R1,0
Biochar,16,50,180,12,50,0,0,2,0,R2,0
Biochar,16,50,180,12,50,0,0,2,0,R3,0.245098
Biochar,17,50,180,8,20,0,0,0,-1,R1,-5.514706
Biochar,17,50,180,8,20,0,0,0,-1,R2,-3.063725
Biochar,17,50,180,8,20,0,0,0,-1,R3,-4.901961
Biochar,18,50,180,8,100,0,0,0,1,R1,0.122549
Biochar,18,50,180,8,100,0,0,0,1,R2,-0.367647
Biochar,18,50,180,8,100,0,0,0,1,R3,-0.490196
Biochar,19,50,180,8,10,0,0,0,-2,R1,-2.45098
Biochar,19,50,180,8,10,0,0,0,-2,R2,-11.029412
Biochar,19,50,180,8,10,0,0,0,-2,R3,-7.352941
Biochar,20,50,180,8,150,0,0,0,2,R1,-0.163399
Biochar,20,50,180,8,150,0,0,0,2,R2,0.163399
Biochar,20,50,180,8,150,0,0,0,2,R3,-0.081699
Biochar,21,25,120,6,20,-1,-1,-1,-1,R1,-2.144608
Biochar,21,25,120,6,20,-1,-1,-1,-1,R2,-5.514706
Biochar,21,25,120,6,20,-1,-1,-1,-1,R3,-4.289216
Biochar,22,25,240,10,100,-1,1,1,1,R1,-1.041667
Biochar,22,25,240,10,100,-1,1,1,1,R2,-1.102941
Biochar,22,25,240,10,100,-1,1,1,1,R3,-0.735294
Biochar,23,75,120,10,20,1,-1,1,-1,R1,-14.093137
Biochar,23,75,120,10,20,1,-1,1,-1,R2,-13.480392
Biochar,23,75,120,10,20,1,-1,1,-1,R3,-11.642157
Biochar,24,75,240,6,100,1,1,-1,1,R1,-0.612745
Biochar,24,75,240,6,100,1,1,-1,1,R2,0.612745
Biochar,24,75,240,6,100,1,1,-1,1,R3,0.367647
Biochar,25,25,120,10,100,-1,-1,1,1,R1,0.061275
Biochar,25,25,120,10,100,-1,-1,1,1,R2,-0.122549
Biochar,25,25,120,10,100,-1,-1,1,1,R3,-0.061275
Biochar,26,25,240,6,20,-1,1,-1,-1,R1,-3.370098
Biochar,26,25,240,6,20,-1,1,-1,-1,R2,-0.306373
Biochar,26,25,240,6,20,-1,1,-1,-1,R3,-1.838235
Biochar,27,75,120,6,100,1,-1,-1,1,R1,2.083333
Biochar,27,75,120,6,100,1,-1,-1,1,R2,1.348039
Biochar,27,75,120,6,100,1,-1,-1,1,R3,1.715686
Biochar,28,75,240,10,20,1,1,1,-1,R1,-3.063725
Biochar,28,75,240,10,20,1,1,1,-1,R2,-0.612745
Biochar,28,75,240,10,20,1,1,1,-1,R3,-3.676471
Biochar,29,10,60,4,10,-2,-2,-2,-2,R1,-9.436275
Biochar,29,10,60,4,10,-2,-2,-2,-2,R2,-10.04902
Biochar,29,10,60,4,10,-2,-2,-2,-2,R3,-10.04902
Biochar,30,10,360,12,150,-2,2,2,2,R1,0.555556
Biochar,30,10,360,12,150,-2,2,2,2,R2,0.02451
Biochar,30,10,360,12,150,-2,2,2,2,R3,0.392157
Biochar,31,100,60,12,10,2,-2,2,-2,R1,-11.029412
Biochar,31,100,60,12,10,2,-2,2,-2,R2,-17.156863
Biochar,31,100,60,12,10,2,-2,2,-2,R3,-14.705882
Biochar,32,100,360,4,150,2,2,-2,2,R1,4.166667
Biochar,32,100,360,4,150,2,2,-2,2,R2,4.411765
Biochar,32,100,360,4,150,2,2,-2,2,R3,3.921569
Repyrolysis 1:1,1,50,180,8,50,0,0,0,0,R1,35.294118
Repyrolysis 1:1,1,50,180,8,50,0,0,0,0,R2,35.539216
Repyrolysis 1:1,1,50,180,8,50,0,0,0,0,R3,35.294118
Repyrolysis 1:1,2,50,180,8,50,0,0,0,0,R1,34.313725
Repyrolysis 1:1,2,50,180,8,50,0,0,0,0,R2,35.784314
Repyrolysis 1:1,2,50,180,8,50,0,0,0,0,R3,34.558824
Repyrolysis 1:1,3,50,180,8,50,0,0,0,0,R1,33.823529
Repyrolysis 1:1,3,50,180,8,50,0,0,0,0,R2,34.558824
Repyrolysis 1:1,3,50,180,8,50,0,0,0,0,R3,33.333333
Repyrolysis 1:1,4,50,180,8,50,0,0,0,0,R1,35.04902
Repyrolysis 1:1,4,50,180,8,50,0,0,0,0,R2,33.823529
Repyrolysis 1:1,4,50,180,8,50,0,0,0,0,R3,36.029412
Repyrolysis 1:1,5,25,180,8,50,-1,0,0,0,R1,23.406863
Repyrolysis 1:1,5,25,180,8,50,-1,0,0,0,R2,22.794118
Repyrolysis 1:1,5,25,180,8,50,-1,0,0,0,R3,22.54902
Repyrolysis 1:1,6,75,180,8,50,1,0,0,0,R1,40.196078
Repyrolysis 1:1,6,75,180,8,50,1,0,0,0,R2,36.27451
Repyrolysis 1:1,6,75,180,8,50,1,0,0,0,R3,37.745098
Repyrolysis 1:1,7,10,180,8,50,-2,0,0,0,R1,9.509804
Repyrolysis 1:1,7,10,180,8,50,-2,0,0,0,R2,9.387255
Repyrolysis 1:1,7,10,180,8,50,-2,0,0,0,R3,9.632353
Repyrolysis 1:1,8,100,180,8,50,2,0,0,0,R1,54.656863
Repyrolysis 1:1,8,100,180,8,50,2,0,0,0,R2,49.754902
Repyrolysis 1:1,8,100,180,8,50,2,0,0,0,R3,48.284314
Repyrolysis 1:1,9,50,120,8,50,0,-1,0,0,R1,27.696078
Repyrolysis 1:1,9,50,120,8,50,0,-1,0,0,R2,28.431373
Repyrolysis 1:1,9,50,120,8,50,0,-1,0,0,R3,29.166667
Repyrolysis 1:1,10,50,240,8,50,0,1,0,0,R1,35.784314
Repyrolysis 1:1,10,50,240,8,50,0,1,0,0,R2,36.764706
Repyrolysis 1:1,10,50,240,8,50,0,1,0,0,R3,34.558824
Repyrolysis 1:1,11,50,60,8,50,0,-2,0,0,R1,22.794118
Repyrolysis 1:1,11,50,60,8,50,0,-2,0,0,R2,21.813725
Repyrolysis 1:1,11,50,60,8,50,0,-2,0,0,R3,21.078431
Repyrolysis 1:1,12,50,360,8,50,0,2,0,0,R1,35.784314
Repyrolysis 1:1,12,50,360,8,50,0,2,0,0,R2,36.27451
Repyrolysis 1:1,12,50,360,8,50,0,2,0,0,R3,37.009804
Repyrolysis 1:1,13,50,180,6,50,0,0,-1,0,R1,40.196078
Repyrolysis 1:1,13,50,180,6,50,0,0,-1,0,R2,40.686275
Repyrolysis 1:1,13,50,180,6,50,0,0,-1,0,R3,39.460784
Repyrolysis 1:1,14,50,180,10,50,0,0,1,0,R1,32.843137
Repyrolysis 1:1,14,50,180,10,50,0,0,1,0,R2,34.558824
Repyrolysis 1:1,14,50,180,10,50,0,0,1,0,R3,35.294118
Repyrolysis 1:1,15,50,180,4,50,0,0,-2,0,R1,44.852941
Repyrolysis 1:1,15,50,180,4,50,0,0,-2,0,R2,45.098039
Repyrolysis 1:1,15,50,180,4,50,0,0,-2,0,R3,45.833333
Repyrolysis 1:1,16,50,180,12,50,0,0,2,0,R1,21.568627
Repyrolysis 1:1,16,50,180,12,50,0,0,2,0,R2,21.078431
Repyrolysis 1:1,16,50,180,12,50,0,0,2,0,R3,21.323529
Repyrolysis 1:1,17,50,180,8,20,0,0,0,-1,R1,37.377451
Repyrolysis 1:1,17,50,180,8,20,0,0,0,-1,R2,39.828431
Repyrolysis 1:1,17,50,180,8,20,0,0,0,-1,R3,41.053922
Repyrolysis 1:1,18,50,180,8,100,0,0,0,1,R1,24.264706
Repyrolysis 1:1,18,50,180,8,100,0,0,0,1,R2,23.406863
Repyrolysis 1:1,18,50,180,8,100,0,0,0,1,R3,24.019608
Repyrolysis 1:1,19,50,180,8,10,0,0,0,-2,R1,42.892157
Repyrolysis 1:1,19,50,180,8,10,0,0,0,-2,R2,42.892157
Repyrolysis 1:1,19,50,180,8,10,0,0,0,-2,R3,44.117647
Repyrolysis 1:1,20,50,180,8,150,0,0,0,2,R1,16.503268
Repyrolysis 1:1,20,50,180,8,150,0,0,0,2,R2,16.584967
Repyrolysis 1:1,20,50,180,8,150,0,0,0,2,R3,16.094771
Repyrolysis 1:1,21,25,120,6,20,-1,-1,-1,-1,R1,33.394608
Repyrolysis 1:1,21,25,120,6,20,-1,-1,-1,-1,R2,27.879902
Repyrolysis 1:1,21,25,120,6,20,-1,-1,-1,-1,R3,28.79902
Repyrolysis 1:1,22,25,240,10,100,-1,1,1,1,R1,11.029412
Repyrolysis 1:1,22,25,240,10,100,-1,1,1,1,R2,10.968137
Repyrolysis 1:1,22,25,240,10,100,-1,1,1,1,R3,11.151961
Repyrolysis 1:1,23,75,120,10,20,1,-1,1,-1,R1,56.372549
Repyrolysis 1:1,23,75,120,10,20,1,-1,1,-1,R2,42.279412
Repyrolysis 1:1,23,75,120,10,20,1,-1,1,-1,R3,53.308824
Repyrolysis 1:1,24,75,240,6,100,1,1,-1,1,R1,34.791667
Repyrolysis 1:1,24,75,240,6,100,1,1,-1,1,R2,36.25
Repyrolysis 1:1,24,75,240,6,100,1,1,-1,1,R3,35.729167
Repyrolysis 1:1,25,25,120,10,100,-1,-1,1,1,R1,10.171569
Repyrolysis 1:1,25,25,120,10,100,-1,-1,1,1,R2,10.416667
Repyrolysis 1:1,25,25,120,10,100,-1,-1,1,1,R3,9.375
Repyrolysis 1:1,26,25,240,6,20,-1,1,-1,-1,R1,53.308824
Repyrolysis 1:1,26,25,240,6,20,-1,1,-1,-1,R2,53.308824
Repyrolysis 1:1,26,25,240,6,20,-1,1,-1,-1,R3,56.678922
Repyrolysis 1:1,27,75,120,6,100,1,-1,-1,1,R1,32.107843
Repyrolysis 1:1,27,75,120,6,100,1,-1,-1,1,R2,34.926471
Repyrolysis 1:1,27,75,120,6,100,1,-1,-1,1,R3,34.068627
Repyrolysis 1:1,28,75,240,10,20,1,1,1,-1,R1,36.151961
Repyrolysis 1:1,28,75,240,10,20,1,1,1,-1,R2,41.053922
Repyrolysis 1:1,28,75,240,10,20,1,1,1,-1,R3,39.215686
Repyrolysis 1:1,29,10,60,4,10,-2,-2,-2,-2,R1,46.323529
Repyrolysis 1:1,29,10,60,4,10,-2,-2,-2,-2,R2,46.936275
Repyrolysis 1:1,29,10,60,4,10,-2,-2,-2,-2,R3,46.323529
Repyrolysis 1:1,30,10,360,12,150,-2,2,2,2,R1,3.129085
Repyrolysis 1:1,30,10,360,12,150,-2,2,2,2,R2,3.169935
Repyrolysis 1:1,30,10,360,12,150,-2,2,2,2,R3,3.129085
Repyrolysis 1:1,31,100,60,12,10,2,-2,2,-2,R1,56.372549
Repyrolysis 1:1,31,100,60,12,10,2,-2,2,-2,R2,58.823529
Repyrolysis 1:1,31,100,60,12,10,2,-2,2,-2,R3,61.27451
Repyrolysis 1:1,32,100,360,4,150,2,2,-2,2,R1,32.271242
Repyrolysis 1:1,32,100,360,4,150,2,2,-2,2,R2,32.51634
Repyrolysis 1:1,32,100,360,4,150,2,2,-2,2,R3,32.598039
Repyrolysis 2:1,1,50,180,8,50,0,0,0,0,R1,35.294118
Repyrolysis 2:1,1,50,180,8,50,0,0,0,0,R2,35.539216
Repyrolysis 2:1,1,50,180,8,50,0,0,0,0,R3,35.294118
Repyrolysis 2:1,2,50,180,8,50,0,0,0,0,R1,34.313725
Repyrolysis 2:1,2,50,180,8,50,0,0,0,0,R2,35.784314
Repyrolysis 2:1,2,50,180,8,50,0,0,0,0,R3,34.558824
Repyrolysis 2:1,3,50,180,8,50,0,0,0,0,R1,33.823529
Repyrolysis 2:1,3,50,180,8,50,0,0,0,0,R2,34.558824
Repyrolysis 2:1,3,50,180,8,50,0,0,0,0,R3,33.333333
Repyrolysis 2:1,4,50,180,8,50,0,0,0,0,R1,35.04902
Repyrolysis 2:1,4,50,180,8,50,0,0,0,0,R2,33.823529
Repyrolysis 2:1,4,50,180,8,50,0,0,0,0,R3,36.029412
Repyrolysis 2:1,5,25,180,8,50,-1,0,0,0,R1,23.406863
Repyrolysis 2:1,5,25,180,8,50,-1,0,0,0,R2,22.794118
Repyrolysis 2:1,5,25,180,8,50,-1,0,0,0,R3,22.54902
Repyrolysis 2:1,6,75,180,8,50,1,0,0,0,R1,40.196078
Repyrolysis 2:1,6,75,180,8,50,1,0,0,0,R2,36.27451
Repyrolysis 2:1,6,75,180,8,50,1,0,0,0,R3,37.745098
Repyrolysis 2:1,7,10,180,8,50,-2,0,0,0,R1,9.509804
Repyrolysis 2:1,7,10,180,8,50,-2,0,0,0,R2,9.387255
Repyrolysis 2:1,7,10,180,8,50,-2,0,0,0,R3,9.632353
Repyrolysis 2:1,8,100,180,8,50,2,0,0,0,R1,54.656863
Repyrolysis 2:1,8,100,180,8,50,2,0,0,0,R2,49.754902
Repyrolysis 2:1,8,100,180,8,50,2,0,0,0,R3,48.284314
Repyrolysis 2:1,9,50,120,8,50,0,-1,0,0,R1,27.696078
Repyrolysis 2:1,9,50,120,8,50,0,-1,0,0,R2,28.431373
Repyrolysis 2:1,9,50,120,8,50,0,-1,0,0,R3,29.166667
Repyrolysis 2:1,10,50,240,8,50,0,1,0,0,R1,35.784314
Repyrolysis 2:1,10,50,240,8,50,0,1,0,0,R2,36.764706
Repyrolysis 2:1,10,50,240,8,50,0,1,0,0,R3,34.558824
Repyrolysis 2:1,11,50,60,8,50,0,-2,0,0,R1,22.794118
Repyrolysis 2:1,11,50,60,8,50,0,-2,0,0,R2,21.813725
Repyrolysis 2:1,11,50,60,8,50,0,-2,0,0,R3,21.078431
Repyrolysis 2:1,12,50,360,8,50,0,2,0,0,R1,35.784314
Repyrolysis 2:1,12,50,360,8,50,0,2,0,0,R2,36.27451
Repyrolysis 2:1,12,50,360,8,50,0,2,0,0,R3,37.009804
Repyrolysis 2:1,13,50,180,6,50,0,0,-1,0,R1,40.196078
Repyrolysis 2:1,13,50,180,6,50,0,0,-1,0,R2,40.686275
Repyrolysis 2:1,13,50,180,6,50,0,0,-1,0,R3,39.460784
Repyrolysis 2:1,14,50,180,10,50,0,0,1,0,R1,32.843137
Repyrolysis 2:1,14,50,180,10,50,0,0,1,0,R2,34.558824
Repyrolysis 2:1,14,50,180,10,50,0,0,1,0,R3,35.294118
Repyrolysis 2:1,15,50,180,4,50,0,0,-2,0,R1,44.852941
Repyrolysis 2:1,15,50,180,4,50,0,0,-2,0,R2,45.098039
Repyrolysis 2:1,15,50,180,4,50,0,0,-2,0,R3,45.833333
Repyrolysis 2:1,16,50,180,12,50,0,0,2,0,R1,21.568627
Repyrolysis 2:1,16,50,180,12,50,0,0,2,0,R2,21.078431
Repyrolysis 2:1,16,50,180,12,50,0,0,2,0,R3,21.323529
Repyrolysis 2:1,17,50,180,8,20,0,0,0,-1,R1,37.377451
Repyrolysis 2:1,17,50,180,8,20,0,0,0,-1,R2,39.828431
Repyrolysis 2:1,17,50,180,8,20,0,0,0,-1,R3,41.053922
Repyrolysis 2:1,18,50,180,8,100,0,0,0,1,R1,24.264706
Repyrolysis 2:1,18,50,180,8,100,0,0,0,1,R2,23.406863
Repyrolysis 2:1,18,50,180,8,100,0,0,0,1,R3,24.019608
Repyrolysis 2:1,19,50,180,8,10,0,0,0,-2,R1,42.892157
Repyrolysis 2:1,19,50,180,8,10,0,0,0,-2,R2,42.892157
Repyrolysis 2:1,19,50,180,8,10,0,0,0,-2,R3,44.117647
Repyrolysis 2:1,20,50,180,8,150,0,0,0,2,R1,16.503268
Repyrolysis 2:1,20,50,180,8,150,0,0,0,2,R2,16.584967
Repyrolysis 2:1,20,50,180,8,150,0,0,0,2,R3,16.094771
Repyrolysis 2:1,21,25,120,6,20,-1,-1,-1,-1,R1,33.394608
Repyrolysis 2:1,21,25,120,6,20,-1,-1,-1,-1,R2,27.879902
Repyrolysis 2:1,21,25,120,6,20,-1,-1,-1,-1,R3,28.79902
Repyrolysis 2:1,22,25,240,10,100,-1,1,1,1,R1,11.029412
Repyrolysis 2:1,22,25,240,10,100,-1,1,1,1,R2,10.968137
Repyrolysis 2:1,22,25,240,10,100,-1,1,1,1,R3,11.151961
Repyrolysis 2:1,23,75,120,10,20,1,-1,1,-1,R1,56.372549
Repyrolysis 2:1,23,75,120,10,20,1,-1,1,-1,R2,42.279412
Repyrolysis 2:1,23,75,120,10,20,1,-1,1,-1,R3,53.308824
Repyrolysis 2:1,24,75,240,6,100,1,1,-1,1,R1,34.791667
Repyrolysis 2:1,24,75,240,6,100,1,1,-1,1,R2,36.25
Repyrolysis 2:1,24,75,240,6,100,1,1,-1,1,R3,35.729167
Repyrolysis 2:1,25,25,120,10,100,-1,-1,1,1,R1,10.171569
Repyrolysis 2:1,25,25,120,10,100,-1,-1,1,1,R2,10.416667
Repyrolysis 2:1,25,25,120,10,100,-1,-1,1,1,R3,9.375
Repyrolysis 2:1,26,25,240,6,20,-1,1,-1,-1,R1,53.308824
Repyrolysis 2:1,26,25,240,6,20,-1,1,-1,-1,R2,53.308824
Repyrolysis 2:1,26,25,240,6,20,-1,1,-1,-1,R3,56.678922
Repyrolysis 2:1,27,75,120,6,100,1,-1,-1,1,R1,32.107843
Repyrolysis 2:1,27,75,120,6,100,1,-1,-1,1,R2,34.926471
Repyrolysis 2:1,27,75,120,6,100,1,-1,-1,1,R3,34.068627
Repyrolysis 2:1,28,75,240,10,20,1,1,1,-1,R1,36.151961
Repyrolysis 2:1,28,75,240,10,20,1,1,1,-1,R2,41.053922
Repyrolysis 2:1,28,75,240,10,20,1,1,1,-1,R3,39.215686
Repyrolysis 2:1,29,10,60,4,10,-2,-2,-2,-2,R1,46.323529
Repyrolysis 2:1,29,10,60,4,10,-2,-2,-2,-2,R2,46.936275
Repyrolysis 2:1,29,10,60,4,10,-2,-2,-2,-2,R3,46.323529
Repyrolysis 2:1,30,10,360,12,150,-2,2,2,2,R1,3.129085
Repyrolysis 2:1,30,10,360,12,150,-2,2,2,2,R2,3.169935
Repyrolysis 2:1,30,10,360,12,150,-2,2,2,2,R3,3.129085
Repyrolysis 2:1,31,100,60,12,10,2,-2,2,-2,R1,56.372549
Repyrolysis 2:1,31,100,60,12,10,2,-2,2,-2,R2,49.2
Repyrolysis 2:1,31,100,60,12,10,2,-2,2,-2,R3,49.019608
Repyrolysis 2:1,32,100,360,4,150,2,2,-2,2,R1,32.271242
Repyrolysis 2:1,32,100,360,4,150,2,2,-2,2,R2,32.51634
Repyrolysis 2:1,32,100,360,4,150,2,2,-2,2,R3,32.598039
BC:MgCO3 1:1,1,50,180,8,50,0,0,0,0,R1,41.421569
BC:MgCO3 1:1,1,50,180,8,50,0,0,0,0,R2,41.176471
BC:MgCO3 1:1,1,50,180,8,50,0,0,0,0,R3,40.441176
BC:MgCO3 1:1,2,50,180,8,50,0,0,0,0,R1,39.460784
BC:MgCO3 1:1,2,50,180,8,50,0,0,0,0,R2,41.176471
BC:MgCO3 1:1,2,50,180,8,50,0,0,0,0,R3,40.686275
BC:MgCO3 1:1,3,50,180,8,50,0,0,0,0,R1,41.666667
BC:MgCO3 1:1,3,50,180,8,50,0,0,0,0,R2,41.666667
BC:MgCO3 1:1,3,50,180,8,50,0,0,0,0,R3,41.421569
BC:MgCO3 1:1,4,50,180,8,50,0,0,0,0,R1,41.421569
BC:MgCO3 1:1,4,50,180,8,50,0,0,0,0,R2,41.911765
BC:MgCO3 1:1,4,50,180,8,50,0,0,0,0,R3,42.401961
BC:MgCO3 1:1,5,25,180,8,50,-1,0,0,0,R1,21.078431
BC:MgCO3 1:1,5,25,180,8,50,-1,0,0,0,R2,20.710784
BC:MgCO3 1:1,5,25,180,8,50,-1,0,0,0,R3,20.833333
BC:MgCO3 1:1,6,75,180,8,50,1,0,0,0,R1,48.529412
BC:MgCO3 1:1,6,75,180,8,50,1,0,0,0,R2,47.303922
BC:MgCO3 1:1,6,75,180,8,50,1,0,0,0,R3,40.441176
BC:MgCO3 1:1,7,10,180,8,50,-2,0,0,0,R1,8.529412
BC:MgCO3 1:1,7,10,180,8,50,-2,0,0,0,R2,8.77451
BC:MgCO3 1:1,7,10,180,8,50,-2,0,0,0,R3,8.77451
BC:MgCO3 1:1,8,100,180,8,50,2,0,0,0,R1,65.441176
BC:MgCO3 1:1,8,100,180,8,50,2,0,0,0,R2,66.176471
BC:MgCO3 1:1,8,100,180,8,50,2,0,0,0,R3,64.95098
BC:MgCO3 1:1,9,50,120,8,50,0,-1,0,0,R1,40.196078
BC:MgCO3 1:1,9,50,120,8,50,0,-1,0,0,R2,39.705882
BC:MgCO3 1:1,9,50,120,8,50,0,-1,0,0,R3,40.441176
BC:MgCO3 1:1,10,50,240,8,50,0,1,0,0,R1,42.401961
BC:MgCO3 1:1,10,50,240,8,50,0,1,0,0,R2,41.911765
BC:MgCO3 1:1,10,50,240,8,50,0,1,0,0,R3,43.137255
BC:MgCO3 1:1,11,50,60,8,50,0,-2,0,0,R1,35.784314
BC:MgCO3 1:1,11,50,60,8,50,0,-2,0,0,R2,34.558824
BC:MgCO3 1:1,11,50,60,8,50,0,-2,0,0,R3,32.843137
BC:MgCO3 1:1,12,50,360,8,50,0,2,0,0,R1,44.852941
BC:MgCO3 1:1,12,50,360,8,50,0,2,0,0,R2,43.382353
BC:MgCO3 1:1,12,50,360,8,50,0,2,0,0,R3,46.078431
BC:MgCO3 1:1,13,50,180,6,50,0,0,-1,0,R1,44.852941
BC:MgCO3 1:1,13,50,180,6,50,0,0,-1,0,R2,44.117647
BC:MgCO3 1:1,13,50,180,6,50,0,0,-1,0,R3,45.343137
BC:MgCO3 1:1,14,50,180,10,50,0,0,1,0,R1,34.313725
BC:MgCO3 1:1,14,50,180,10,50,0,0,1,0,R2,34.803922
BC:MgCO3 1:1,14,50,180,10,50,0,0,1,0,R3,34.068627
BC:MgCO3 1:1,15,50,180,4,50,0,0,-2,0,R1,46.323529
BC:MgCO3 1:1,15,50,180,4,50,0,0,-2,0,R2,45.588235
BC:MgCO3 1:1,15,50,180,4,50,0,0,-2,0,R3,44.607843
BC:MgCO3 1:1,16,50,180,12,50,0,0,2,0,R1,28.921569
BC:MgCO3 1:1,16,50,180,12,50,0,0,2,0,R2,29.411765
BC:MgCO3 1:1,16,50,180,12,50,0,0,2,0,R3,27.696078
BC:MgCO3 1:1,17,50,180,8,20,0,0,0,-1,R1,77.818627
BC:MgCO3 1:1,17,50,180,8,20,0,0,0,-1,R2,74.142157
BC:MgCO3 1:1,17,50,180,8,20,0,0,0,-1,R3,82.107843
BC:MgCO3 1:1,18,50,180,8,100,0,0,0,1,R1,22.916667
BC:MgCO3 1:1,18,50,180,8,100,0,0,0,1,R2,23.039216
BC:MgCO3 1:1,18,50,180,8,100,0,0,0,1,R3,23.039216
BC:MgCO3 1:1,19,50,180,8,10,0,0,0,-2,R1,98.039216
BC:MgCO3 1:1,19,50,180,8,10,0,0,0,-2,R2,91.911765
BC:MgCO3 1:1,19,50,180,8,10,0,0,0,-2,R3,84.558824
BC:MgCO3 1:1,20,50,180,8,150,0,0,0,2,R1,15.849673
BC:MgCO3 1:1,20,50,180,8,150,0,0,0,2,R2,15.767974
BC:MgCO3 1:1,20,50,180,8,150,0,0,0,2,R3,16.094771
BC:MgCO3 1:1,21,25,120,6,20,-1,-1,-1,-1,R1,49.632353
BC:MgCO3 1:1,21,25,120,6,20,-1,-1,-1,-1,R2,50.857843
BC:MgCO3 1:1,21,25,120,6,20,-1,-1,-1,-1,R3,49.938725
BC:MgCO3 1:1,22,25,240,10,100,-1,1,1,1,R1,11.764706
BC:MgCO3 1:1,22,25,240,10,100,-1,1,1,1,R2,12.071078
BC:MgCO3 1:1,22,25,240,10,100,-1,1,1,1,R3,11.887255
BC:MgCO3 1:1,23,75,120,10,20,1,-1,1,-1,R1,64.338235
BC:MgCO3 1:1,23,75,120,10,20,1,-1,1,-1,R2,62.5
BC:MgCO3 1:1,23,75,120,10,20,1,-1,1,-1,R3,65.563725
BC:MgCO3 1:1,24,75,240,6,100,1,1,-1,1,R1,35.539216
BC:MgCO3 1:1,24,75,240,6,100,1,1,-1,1,R2,34.681373
BC:MgCO3 1:1,24,75,240,6,100,1,1,-1,1,R3,35.416667
BC:MgCO3 1:1,25,25,120,10,100,-1,-1,1,1,R1,11.090686
BC:MgCO3 1:1,25,25,120,10,100,-1,-1,1,1,R2,11.27451
BC:MgCO3 1:1,25,25,120,10,100,-1,-1,1,1,R3,11.397059
BC:MgCO3 1:1,26,25,240,6,20,-1,1,-1,-1,R1,57.291667
BC:MgCO3 1:1,26,25,240,6,20,-1,1,-1,-1,R2,50.857843
BC:MgCO3 1:1,26,25,240,6,20,-1,1,-1,-1,R3,51.164216
BC:MgCO3 1:1,27,75,120,6,100,1,-1,-1,1,R1,35.04902
BC:MgCO3 1:1,27,75,120,6,100,1,-1,-1,1,R2,35.661765
BC:MgCO3 1:1,27,75,120,6,100,1,-1,-1,1,R3,36.029412
BC:MgCO3 1:1,28,75,240,10,20,1,1,1,-1,R1,82.107843
BC:MgCO3 1:1,28,75,240,10,20,1,1,1,-1,R2,83.333333
BC:MgCO3 1:1,28,75,240,10,20,1,1,1,-1,R3,85.784314
BC:MgCO3 1:1,29,10,60,4,10,-2,-2,-2,-2,R1,24.264706
BC:MgCO3 1:1,29,10,60,4,10,-2,-2,-2,-2,R2,25.490196
BC:MgCO3 1:1,29,10,60,4,10,-2,-2,-2,-2,R3,23.039216
BC:MgCO3 1:1,30,10,360,12,150,-2,2,2,2,R1,2.924837
BC:MgCO3 1:1,30,10,360,12,150,-2,2,2,2,R2,2.965686
BC:MgCO3 1:1,30,10,360,12,150,-2,2,2,2,R3,2.965686
BC:MgCO3 1:1,31,100,60,12,10,2,-2,2,-2,R1,66.176471
BC:MgCO3 1:1,31,100,60,12,10,2,-2,2,-2,R2,66.176471
BC:MgCO3 1:1,31,100,60,12,10,2,-2,2,-2,R3,77.205882
BC:MgCO3 1:1,32,100,360,4,150,2,2,-2,2,R1,31.617647
BC:MgCO3 1:1,32,100,360,4,150,2,2,-2,2,R2,32.434641
BC:MgCO3 1:1,32,100,360,4,150,2,2,-2,2,R3,32.189542
BC:MgCO3 2:1,1,50,180,8,50,0,0,0,0,R1,19.117647
BC:MgCO3 2:1,1,50,180,8,50,0,0,0,0,R2,17.892157
BC:MgCO3 2:1,1,50,180,8,50,0,0,0,0,R3,19.607843
BC:MgCO3 2:1,2,50,180,8,50,0,0,0,0,R1,21.813725
BC:MgCO3 2:1,2,50,180,8,50,0,0,0,0,R2,22.058824
BC:MgCO3 2:1,2,50,180,8,50,0,0,0,0,R3,20.343137
BC:MgCO3 2:1,3,50,180,8,50,0,0,0,0,R1,19.117647
BC:MgCO3 2:1,3,50,180,8,50,0,0,0,0,R2,18.872549
BC:MgCO3 2:1,3,50,180,8,50,0,0,0,0,R3,17.647059
BC:MgCO3 2:1,4,50,180,8,50,0,0,0,0,R1,22.303922
BC:MgCO3 2:1,4,50,180,8,50,0,0,0,0,R2,21.813725
BC:MgCO3 2:1,4,50,180,8,50,0,0,0,0,R3,21.813725
BC:MgCO3 2:1,5,25,180,8,50,-1,0,0,0,R1,9.558824
BC:MgCO3 2:1,5,25,180,8,50,-1,0,0,0,R2,8.946078
BC:MgCO3 2:1,5,25,180,8,50,-1,0,0,0,R3,9.803922
BC:MgCO3 2:1,6,75,180,8,50,1,0,0,0,R1,27.45098
BC:MgCO3 2:1,6,75,180,8,50,1,0,0,0,R2,31.862745
BC:MgCO3 2:1,6,75,180,8,50,1,0,0,0,R3,35.294118
BC:MgCO3 2:1,7,10,180,8,50,-2,0,0,0,R1,6.20098
BC:MgCO3 2:1,7,10,180,8,50,-2,0,0,0,R2,4.362745
BC:MgCO3 2:1,7,10,180,8,50,-2,0,0,0,R3,5.955882
BC:MgCO3 2:1,8,100,180,8,50,2,0,0,0,R1,26.960784
BC:MgCO3 2:1,8,100,180,8,50,2,0,0,0,R2,32.598039
BC:MgCO3 2:1,8,100,180,8,50,2,0,0,0,R3,31.372549
BC:MgCO3 2:1,9,50,120,8,50,0,-1,0,0,R1,18.137255
BC:MgCO3 2:1,9,50,120,8,50,0,-1,0,0,R2,19.362745
BC:MgCO3 2:1,9,50,120,8,50,0,-1,0,0,R3,20.098039
BC:MgCO3 2:1,10,50,240,8,50,0,1,0,0,R1,28.186275
BC:MgCO3 2:1,10,50,240,8,50,0,1,0,0,R2,31.127451
BC:MgCO3 2:1,10,50,240,8,50,0,1,0,0,R3,28.676471
BC:MgCO3 2:1,11,50,60,8,50,0,-2,0,0,R1,14.215686
BC:MgCO3 2:1,11,50,60,8,50,0,-2,0,0,R2,15.686275
BC:MgCO3 2:1,11,50,60,8,50,0,-2,0,0,R3,13.970588
BC:MgCO3 2:1,12,50,360,8,50,0,2,0,0,R1,32.843137
BC:MgCO3 2:1,12,50,360,8,50,0,2,0,0,R2,32.843137
BC:MgCO3 2:1,12,50,360,8,50,0,2,0,0,R3,33.333333
BC:MgCO3 2:1,13,50,180,6,50,0,0,-1,0,R1,15.196078
BC:MgCO3 2:1,13,50,180,6,50,0,0,-1,0,R2,15.686275
BC:MgCO3 2:1,13,50,180,6,50,0,0,-1,0,R3,14.705882
BC:MgCO3 2:1,14,50,180,10,50,0,0,1,0,R1,19.117647
BC:MgCO3 2:1,14,50,180,10,50,0,0,1,0,R2,18.627451
BC:MgCO3 2:1,14,50,180,10,50,0,0,1,0,R3,17.892157
BC:MgCO3 2:1,15,50,180,4,50,0,0,-2,0,R1,29.411765
BC:MgCO3 2:1,15,50,180,4,50,0,0,-2,0,R2,28.676471
BC:MgCO3 2:1,15,50,180,4,50,0,0,-2,0,R3,27.696078
BC:MgCO3 2:1,16,50,180,12,50,0,0,2,0,R1,15.441176
BC:MgCO3 2:1,16,50,180,12,50,0,0,2,0,R2,18.627451
BC:MgCO3 2:1,16,50,180,12,50,0,0,2,0,R3,18.872549
BC:MgCO3 2:1,17,50,180,8,20,0,0,0,-1,R1,27.573529
BC:MgCO3 2:1,17,50,180,8,20,0,0,0,-1,R2,19.607843
BC:MgCO3 2:1,17,50,180,8,20,0,0,0,-1,R3,25.735294
BC:MgCO3 2:1,18,50,180,8,100,0,0,0,1,R1,21.997549
BC:MgCO3 2:1,18,50,180,8,100,0,0,0,1,R2,22.610294
BC:MgCO3 2:1,18,50,180,8,100,0,0,0,1,R3,22.242647
BC:MgCO3 2:1,19,50,180,8,10,0,0,0,-2,R1,35.539216
BC:MgCO3 2:1,19,50,180,8,10,0,0,0,-2,R2,41.666667
BC:MgCO3 2:1,19,50,180,8,10,0,0,0,-2,R3,36.764706
BC:MgCO3 2:1,20,50,180,8,150,0,0,0,2,R1,16.339869
BC:MgCO3 2:1,20,50,180,8,150,0,0,0,2,R2,16.013072
BC:MgCO3 2:1,20,50,180,8,150,0,0,0,2,R3,16.094771
BC:MgCO3 2:1,21,25,120,6,20,-1,-1,-1,-1,R1,0.919118
BC:MgCO3 2:1,21,25,120,6,20,-1,-1,-1,-1,R2,3.676471
BC:MgCO3 2:1,21,25,120,6,20,-1,-1,-1,-1,R3,3.063725
BC:MgCO3 2:1,22,25,240,10,100,-1,1,1,1,R1,12.132353
BC:MgCO3 2:1,22,25,240,10,100,-1,1,1,1,R2,12.071078
BC:MgCO3 2:1,22,25,240,10,100,-1,1,1,1,R3,11.887255
BC:MgCO3 2:1,23,75,120,10,20,1,-1,1,-1,R1,14.705882
BC:MgCO3 2:1,23,75,120,10,20,1,-1,1,-1,R2,13.480392
BC:MgCO3 2:1,23,75,120,10,20,1,-1,1,-1,R3,19.607843
BC:MgCO3 2:1,24,75,240,6,100,1,1,-1,1,R1,30.514706
BC:MgCO3 2:1,24,75,240,6,100,1,1,-1,1,R2,31.004902
BC:MgCO3 2:1,24,75,240,6,100,1,1,-1,1,R3,29.901961
BC:MgCO3 2:1,25,25,120,10,100,-1,-1,1,1,R1,11.887255
BC:MgCO3 2:1,25,25,120,10,100,-1,-1,1,1,R2,12.316176
BC:MgCO3 2:1,25,25,120,10,100,-1,-1,1,1,R3,12.254902
BC:MgCO3 2:1,26,25,240,6,20,-1,1,-1,-1,R1,9.497549
BC:MgCO3 2:1,26,25,240,6,20,-1,1,-1,-1,R2,9.803922
BC:MgCO3 2:1,26,25,240,6,20,-1,1,-1,-1,R3,4.901961
BC:MgCO3 2:1,27,75,120,6,100,1,-1,-1,1,R1,32.352941
BC:MgCO3 2:1,27,75,120,6,100,1,-1,-1,1,R2,34.681373
BC:MgCO3 2:1,27,75,120,6,100,1,-1,-1,1,R3,33.333333
BC:MgCO3 2:1,28,75,240,10,20,1,1,1,-1,R1,22.058824
BC:MgCO3 2:1,28,75,240,10,20,1,1,1,-1,R2,43.504902
BC:MgCO3 2:1,28,75,240,10,20,1,1,1,-1,R3,26.960784
BC:MgCO3 2:1,29,10,60,4,10,-2,-2,-2,-2,R1,1.593137
BC:MgCO3 2:1,29,10,60,4,10,-2,-2,-2,-2,R2,2.205882
BC:MgCO3 2:1,29,10,60,4,10,-2,-2,-2,-2,R3,2.818627
BC:MgCO3 2:1,30,10,360,12,150,-2,2,2,2,R1,3.129085
BC:MgCO3 2:1,30,10,360,12,150,-2,2,2,2,R2,3.129085
BC:MgCO3 2:1,30,10,360,12,150,-2,2,2,2,R3,3.088235
BC:MgCO3 2:1,31,100,60,12,10,2,-2,2,-2,R1,55.147059
BC:MgCO3 2:1,31,100,60,12,10,2,-2,2,-2,R2,49.019608
BC:MgCO3 2:1,31,100,60,12,10,2,-2,2,-2,R3,49.019608
BC:MgCO3 2:1,32,100,360,4,150,2,2,-2,2,R1,31.045752
BC:MgCO3 2:1,32,100,360,4,150,2,2,-2,2,R2,31.699346
BC:MgCO3 2:1,32,100,360,4,150,2,2,-2,2,R3,31.20915"""
d = pd.read_csv(io.StringIO(CSV))
MATS = list(d.mat.unique())
X4 = ['x1', 'x2', 'x3', 'x4']
NAME = {'x1': 'A:Conc', 'x2': 'B:Time', 'x3': 'C:pH', 'x4': 'D:Dose'}
LEVELS = {'x1': ('Concentration (ppm)', [10, 25, 50, 75, 100]), 'x2': ('Time (min)', [60, 120, 180, 240, 360]),
          'x3': ('pH', [4, 6, 8, 10, 12]), 'x4': ('Dosage (mg)', [10, 20, 50, 100, 150])}
to_actual = lambda k, c: np.interp(c, [-2, -1, 0, 1, 2], LEVELS[k][1])
to_coded = lambda k, v: np.interp(v, LEVELS[k][1], [-2, -1, 0, 1, 2])
d['cond'] = np.where(d.run <= 4, 0, d.run)   # centre runs 1-4 = one condition
Y_ORIG = d.y.copy()                          # checked at the end: measured values must stay unchanged
print(d.shape, d.groupby('mat').size().to_dict())

## Model definitions

In [ ]:
LIN = X4
SQ = [f'{a}^2' for a in X4]
INT = [f'{a}*{b}' for i, a in enumerate(X4) for b in X4[i + 1:]]
ALL = LIN + SQ + INT

def design(df, terms):
    X = pd.DataFrame(index=df.index)
    for t in terms:
        if t.endswith('^2'): X[t] = df[t[:-2]] ** 2
        elif '*' in t: a, b = t.split('*'); X[t] = df[a] * df[b]
        else: X[t] = df[t]
    return sm.add_constant(X, has_constant='add')

def terms_of(m): return m.model.exog_names[1:]

def fit_rsm(df, reduce=True, w=None):
    """Quadratic RSM. reduce=True: backward elimination (p > 0.05), keeping hierarchy. w: optional weights (WLS)."""
    ols = lambda X: sm.OLS(df.y, X).fit() if w is None else sm.WLS(df.y, X, weights=w).fit()
    terms = ALL.copy()
    while reduce:
        m = ols(design(df, terms))
        needed = {x for t in terms if t not in LIN for x in X4 if x in t}
        p = m.pvalues[[t for t in terms if t not in needed]]
        if p.empty or p.max() <= 0.05: break
        terms.remove(p.idxmax())
    return ols(design(df, terms))

def rsm_predict(m, df): return m.predict(design(df, terms_of(m))).values

def ann_X(df, cols=X4):
    onehot = pd.get_dummies(df.mat).reindex(columns=MATS, fill_value=0).astype(float)
    return pd.concat([df[cols].astype(float), onehot], axis=1).values

def make_ann(h, a, seed):
    net = MLPRegressor((h,), activation='tanh', solver='lbfgs', alpha=a, max_iter=5000, random_state=seed)
    return TransformedTargetRegressor(make_pipeline(StandardScaler(), net), transformer=StandardScaler())

GRID = [(h, a) for h in (4, 8, 12) for a in (1e-3, 1e-2, 1e-1)]

def fit_ann(df, n_ens=5, cols=X4, target=None, grid=GRID, k=4):
    """One ANN for all materials (material = one-hot input). Neurons and L2 tuned by grouped
    inner CV on THIS data only; the final model is an ensemble of n_ens seeds."""
    X, g = ann_X(df, cols), df.cond.values
    y = df.y.values if target is None else np.asarray(target)
    folds = list(GroupKFold(k).split(X, y, g))
    def cv_mse(h, a):
        return np.mean([mean_squared_error(y[va], make_ann(h, a, SEED).fit(X[tr], y[tr]).predict(X[va])) for tr, va in folds])
    h, a = min(grid, key=lambda p: cv_mse(*p))
    return {'nets': [make_ann(h, a, SEED + i).fit(X, y) for i in range(n_ens)], 'h': h, 'alpha': a, 'cols': cols}

def ann_predict(model, df): return np.mean([n.predict(ann_X(df, model['cols'])) for n in model['nets']], axis=0)

def metrics(a, p): return {'R2': r2_score(a, p), 'RMSE': np.sqrt(mean_squared_error(a, p)), 'MAE': mean_absolute_error(a, p)}

MODELS = ['RSM full', 'RSM reduced', 'ANN']

def score(pred_rows, models=MODELS):
    """Metrics on run means (actual = mean of the 3 replicates), per material and overall."""
    g = pred_rows.groupby(['mat', 'run'])[['y'] + models].mean().reset_index()
    out = [{'Material': m, 'Model': k, **metrics(s.y, s[k])} for m, s in g.groupby('mat', sort=False) for k in models]
    out += [{'Material': 'ALL', 'Model': k, **metrics(g.y, g[k])} for k in models]
    return pd.DataFrame(out).round(3)

## Train / test split (by run condition, no leakage)

In [ ]:
rng = np.random.default_rng(SEED)
# Stratified random draw: 1 run from each factor's axial block + 2 from the factorial block, so training
# keeps information on every factor. Centre (1-4) and extreme corners (29-32) always stay in training.
BLOCKS = [(range(5, 9), 1), (range(9, 13), 1), (range(13, 17), 1), (range(17, 21), 1), (range(21, 29), 2)]
while True:
    test_conds = sorted(int(c) for b, k in BLOCKS for c in rng.choice(list(b), k, replace=False))
    tr0 = d[~d.cond.isin(test_conds) & (d.mat == MATS[0])]
    if np.linalg.matrix_rank(design(tr0, ALL).values) == len(ALL) + 1: break   # full quadratic still estimable
d['set'] = np.where(d.cond.isin(test_conds), 'test', 'train')
train, test = d[d.set == 'train'], d[d.set == 'test']
assert not set(train.cond) & set(test.cond), 'leakage: a condition is in both sets'
print('Test runs (unseen):', test_conds)
print('Train rows:', len(train), '| Test rows:', len(test))
print(d[d.set == 'test'].drop_duplicates('run')[['run', 'conc', 'time', 'pH', 'dose']].to_string(index=False))

## Hold-out test: models fitted on train only, scored once on the unseen test runs

In [ ]:
ann_tr = fit_ann(train)
print(f"ANN chosen on train CV: {ann_tr['h']} hidden neurons, L2 alpha = {ann_tr['alpha']}")
parts = []
for m in MATS:
    tr, te = train[train.mat == m], test[test.mat == m]
    parts.append(te.assign(**{'RSM full': rsm_predict(fit_rsm(tr, False), te),
                              'RSM reduced': rsm_predict(fit_rsm(tr), te),
                              'ANN': ann_predict(ann_tr, te)}))
holdout = pd.concat(parts)
test_metrics = score(holdout)
test_metrics

## Leave-one-condition-out (LOCO): every condition predicted once while unseen
Each fold re-fits the RSM and re-tunes plus re-trains the ANN from scratch (nested), so nothing about the held-out condition leaks in. Takes a few minutes.

In [ ]:
from joblib import Parallel, delayed

def loco_fold(c):
    tr, te = d[d.cond != c], d[d.cond == c]
    ann = fit_ann(tr)
    return pd.concat([te[te.mat == m].assign(**{'RSM full': rsm_predict(fit_rsm(tr[tr.mat == m], False), te[te.mat == m]),
                                               'RSM reduced': rsm_predict(fit_rsm(tr[tr.mat == m]), te[te.mat == m]),
                                               'ANN': ann_predict(ann, te[te.mat == m])}) for m in MATS])

conds = sorted(d.cond.unique())
loco = pd.concat(Parallel(n_jobs=-1, verbose=5)(delayed(loco_fold)(c) for c in conds))   # folds run in parallel
loco_metrics = score(loco)
loco_metrics

## Final RSM per material (all data): ANOVA and fit statistics
The **full** quadratic model shows which linear, square and interaction terms are significant. The **reduced** model (non-significant terms removed, hierarchy kept) is used for optimisation.

In [ ]:
def pretty(t):
    for k, v in NAME.items(): t = t.replace(k, v.split(':')[0])
    return t
def kind(t): return 'Quadratic' if t.endswith('^2') else 'Interaction' if '*' in t else 'Linear'

def anova_table(m, df):
    mse = m.mse_resid
    rows = [('Model', '', m.ess, m.df_model, m.ess / m.df_model, m.fvalue, m.f_pvalue)]
    for t in terms_of(m):
        F = m.tvalues[t] ** 2
        rows.append((f'{pretty(t)} ({t})', kind(t), F * mse, 1, F * mse, F, m.pvalues[t]))
    pe = df.groupby('cond').y.apply(lambda s: ((s - s.mean()) ** 2).sum()).sum()
    dfpe = len(df) - df.cond.nunique()
    lof, dflof = m.ssr - pe, m.df_resid - dfpe
    Flof = (lof / dflof) / (pe / dfpe)
    rows += [('Residual', '', m.ssr, m.df_resid, mse, np.nan, np.nan),
             ('  Lack of fit', '', lof, dflof, lof / dflof, Flof, stats.f.sf(Flof, dflof, dfpe)),
             ('  Pure error', '', pe, dfpe, pe / dfpe, np.nan, np.nan),
             ('Total', '', m.centered_tss, len(df) - 1, np.nan, np.nan, np.nan)]
    t = pd.DataFrame(rows, columns=['Source', 'Term type', 'SS', 'df', 'MS', 'F', 'p-value'])
    t['Significant (p<0.05)'] = np.where(t['p-value'] < 0.05, 'Yes', np.where(t['p-value'].notna(), 'No', ''))
    return t

def fit_stats(m, df):
    h = m.get_influence().hat_matrix_diag
    press = ((m.resid / (1 - h)) ** 2).sum()
    return {'R2': m.rsquared, 'Adj R2': m.rsquared_adj, 'Pred R2': 1 - press / m.centered_tss,
            'Std dev': np.sqrt(m.mse_resid), 'Mean qe': df.y.mean(), 'Model p': m.f_pvalue}

full = {m: fit_rsm(d[d.mat == m], False) for m in MATS}
red = {m: fit_rsm(d[d.mat == m]) for m in MATS}
anova = {m: anova_table(full[m], d[d.mat == m]) for m in MATS}
fit_summary = pd.DataFrame([{'Material': m, 'Model': k, **fit_stats(mod[m], d[d.mat == m])}
                            for m in MATS for k, mod in [('Full quadratic', full), ('Reduced', red)]]).round(4)
for m in MATS:
    a = anova[m]; terms = a[a['Term type'] != '']
    print(f'\n=== {m} ===')
    for k in ['Linear', 'Quadratic', 'Interaction']:
        s = terms[terms['Term type'] == k]
        print(f"  {k:12s} significant: {', '.join(s[s['Significant (p<0.05)'] == 'Yes'].Source) or '-'}")
        print(f"  {'':12s} not sig.   : {', '.join(s[s['Significant (p<0.05)'] == 'No'].Source) or '-'}")
    print('  Reduced model terms:', ', '.join(pretty(t) for t in terms_of(red[m])))
fit_summary

In [ ]:
for m in MATS:
    print(f'\n=== ANOVA (full quadratic): {m} ===')
    display(anova[m].round(4))

## Goal 1: optimisation (maximise qe, pH limited to 4–8)
The search is also restricted to the tested design region (a leverage limit), so the model does not invent qe values at conditions far from any experiment. pH 4–12 is shown only for comparison, to show where a pH-12 "optimum" comes from.

In [ ]:
def optimise(m, df, ph_max=0.0):
    terms = terms_of(m)
    Xd = design(df, terms).values
    XtXi = np.linalg.inv(Xd.T @ Xd)
    hmax = np.einsum('ij,jk,ik->i', Xd, XtXi, Xd).max()
    f = lambda c: design(pd.DataFrame([c], columns=X4), terms).values[0]
    best = None
    for x0 in np.random.default_rng(SEED).uniform(-1, 1, (40, 4)):
        x0[2] = min(x0[2], ph_max)
        r = minimize(lambda c: -f(c) @ m.params.values, x0, method='SLSQP',
                     bounds=[(-2, 2), (-2, 2), (-2, ph_max), (-2, 2)],
                     constraints=[{'type': 'ineq', 'fun': lambda c: hmax - f(c) @ XtXi @ f(c)}])
        if r.success and (best is None or r.fun < best.fun): best = r
    return best.x

def point_report(m, coded, label):
    row = pd.DataFrame([coded], columns=X4)
    pr = m.get_prediction(design(row, terms_of(m))).summary_frame(alpha=0.05).iloc[0]
    return {'Material': label, **{LEVELS[k][0]: round(float(to_actual(k, v)), 1) for k, v in zip(X4, coded)},
            'RSM qe pred': pr['mean'], '95% PI low': pr.obs_ci_lower, '95% PI high': pr.obs_ci_upper,
            'ANN qe pred': float(ann_predict(ann_all, row.assign(mat=label))[0])}

ann_all = fit_ann(d)   # ANN on all data, only to cross-check the RSM optimum
opt_rows, opt_cmp = [], []
for m in MATS:
    df = d[d.mat == m]
    c = optimise(red[m], df, 0.0)
    opt_rows.append({**point_report(red[m], c, m), 'Best observed qe': df.groupby('run').y.mean().max()})
    opt_cmp.append({**point_report(red[m], optimise(red[m], df, 2.0), m)})
optimum = pd.DataFrame(opt_rows).round(2)
optimum_ph4_12 = pd.DataFrame(opt_cmp).round(2)
print('Optimum, pH 4-8 (Biochar = control, shown for completeness only):'); display(optimum)
print('For comparison only, pH 4-12:'); display(optimum_ph4_12)

### Validation plan for the lab
The optimum is rounded to practical lab settings and predicted again. Run each one in the lab (3 replicates); a measured qe inside the 95% prediction interval validates the model.

In [ ]:
STEP = {'x1': 5, 'x2': 10, 'x3': 0.5, 'x4': 5}
val = []
for m in MATS[1:]:
    o = optimum[optimum.Material == m].iloc[0]
    actual = {k: max(LEVELS[k][1][0], round(o[LEVELS[k][0]] / STEP[k]) * STEP[k]) for k in X4}
    coded = [float(to_coded(k, actual[k])) for k in X4]
    val.append(point_report(red[m], coded, m))
validation_plan = pd.DataFrame(val).round(2)
validation_plan

## Goal 2: which material is best (ANOVA)

In [ ]:
two_way = sm.stats.anova_lm(smf.ols('y ~ C(mat) * C(cond)', d).fit(), typ=2)
display(two_way.round(4))
tukey = pairwise_tukeyhsd(d.y, d.mat)
tukey_df = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])
display(tukey_df)

lm = loco_metrics.set_index(['Material', 'Model'])
ranking = pd.DataFrame([{
    'Material': m, 'Mean qe (all runs)': d[d.mat == m].y.mean(), 'Max observed qe': d[d.mat == m].groupby('run').y.mean().max(),
    'Optimum qe (pH 4-8, RSM)': optimum.set_index('Material').loc[m, 'RSM qe pred'],
    'R2': fit_stats(red[m], d[d.mat == m])['R2'], 'Adj R2': fit_stats(red[m], d[d.mat == m])['Adj R2'],
    'Pred R2': fit_stats(red[m], d[d.mat == m])['Pred R2'],
    'LOCO R2 RSM': lm.loc[(m, 'RSM reduced'), 'R2'], 'LOCO R2 ANN': lm.loc[(m, 'ANN'), 'R2']} for m in MATS])
ranking = ranking.sort_values('Mean qe (all runs)', ascending=False).round(3).reset_index(drop=True)
ranking.index += 1
ranking

## Goal 3: actual vs predicted qe for every run
- *RSM fitted*: final reduced model on all data (the classic table in papers).
- *RSM / ANN unseen (LOCO)*: each run predicted by a model that never saw that run, which is the honest accuracy.

In [ ]:
act = d.groupby(['mat', 'run']).agg(conc=('conc', 'first'), time=('time', 'first'), pH=('pH', 'first'), dose=('dose', 'first'),
                                    set=('set', 'first'), actual=('y', 'mean'), actual_sd=('y', 'std')).reset_index()
act['RSM fitted'] = [red[m].predict(design(d[(d.mat == m) & (d.run == r)].iloc[:1], terms_of(red[m]))).iloc[0]
                     for m, r in zip(act.mat, act.run)]
lo = loco.groupby(['mat', 'run'])[['RSM reduced', 'ANN']].mean().rename(columns={'RSM reduced': 'RSM unseen (LOCO)', 'ANN': 'ANN unseen (LOCO)'})
avp = act.merge(lo.reset_index(), on=['mat', 'run']).round(3)
avp['mat'] = pd.Categorical(avp.mat, MATS); avp = avp.sort_values(['mat', 'run'])
# Summary only; the full run-by-run table is in the Excel file (sheet 'Actual vs predicted')
summary = [{'Material': m, **{f'{k} {met}': v for k in ['RSM fitted', 'RSM unseen (LOCO)', 'ANN unseen (LOCO)']
                                for met, v in metrics(s.actual, s[k]).items() if met in ('R2', 'RMSE')}}
           for m, s in avp.groupby('mat', observed=True)]
pd.DataFrame(summary).round(3)

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(24, 5))
for ax, m in zip(axs, MATS):
    s = avp[avp.mat == m]
    for col, mk in [('RSM fitted', 'o'), ('RSM unseen (LOCO)', 's'), ('ANN unseen (LOCO)', '^')]:
        ax.scatter(s.actual, s[col], marker=mk, alpha=.75, label=col)
    lim = [min(s.actual.min(), s[['RSM fitted', 'ANN unseen (LOCO)']].min().min()) - 2,
           max(s.actual.max(), s[['RSM fitted', 'ANN unseen (LOCO)']].max().max()) + 2]
    ax.plot(lim, lim, 'k--', lw=1); ax.set(title=m, xlabel='Actual qe (mg/g)', ylabel='Predicted qe (mg/g)', xlim=lim, ylim=lim)
axs[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT}/actual_vs_predicted.png', dpi=200); plt.show()

## Response surfaces: 6 factor pairs per material
The other two factors are held at that material's pH 4–8 optimum.

In [ ]:
PAIRS = [('x1', 'x2'), ('x1', 'x3'), ('x1', 'x4'), ('x2', 'x3'), ('x2', 'x4'), ('x3', 'x4')]
g = np.linspace(-2, 2, 30)
for m in MATS:
    o = optimum[optimum.Material == m].iloc[0]
    base = {k: float(to_coded(k, o[LEVELS[k][0]])) for k in X4}
    fig = plt.figure(figsize=(20, 11)); fig.suptitle(f'{m}: response surfaces (qe, mg/g)', fontsize=15)
    for i, (a, b) in enumerate(PAIRS):
        A, B = np.meshgrid(g, g)
        grid = pd.DataFrame({k: np.full(A.size, base[k]) for k in X4}); grid[a] = A.ravel(); grid[b] = B.ravel()
        Z = rsm_predict(red[m], grid).reshape(A.shape)
        ax = fig.add_subplot(2, 3, i + 1, projection='3d')
        ax.plot_surface(to_actual(a, A), to_actual(b, B), Z, cmap='viridis', edgecolor='none', alpha=.9)
        ax.set(xlabel=LEVELS[a][0], ylabel=LEVELS[b][0], zlabel='qe (mg/g)', title=f'{LEVELS[a][0]} vs {LEVELS[b][0]}')
    plt.tight_layout(); plt.savefig(f"{OUT}/surface_{m.replace(' ', '_').replace(':', '-')}.png", dpi=150); plt.show()

## Improving accuracy with the same data
**No measured value is changed, removed or added** (checked at the end of this section). Only the way the models read the existing columns changes:
- **conc/dose input:** log(conc ÷ dose), calculated from the existing columns. qe = (C₀ − Ce)·V/m, so qe naturally scales with conc ÷ dose.
- **Removal-space target:** the model learns T = qe × dose ÷ conc (proportional to the fraction removed), and every prediction is converted back exactly: qe = T × conc ÷ dose.
- **Weighted RSM:** conditions with consistent replicates get more weight than noisy ones (weights from training data only).
- **Gaussian Process:** per material, with an automatic relevance length scale for each input.
- **Monotonic boosting:** forced to respect qe ↑ with concentration and qe ↓ with dose (enriched materials; the biochar control is not constrained).
- **RSM + ANN residual:** RSM captures the main trend and an ANN learns what RSM misses.
- **Ensemble:** the average of ANN + conc/dose and GPR + conc/dose, chosen in advance (not picked after seeing the results).

Every model is scored on the **same unseen test runs** and the **same leave-one-condition-out folds** as before, with all tuning done inside the training data.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.ensemble import HistGradientBoostingRegressor
d['logR'] = np.log(d.conc / d.dose)   # input derived from existing columns
train, test = d[d.set == 'train'], d[d.set == 'test']
PHYS = X4 + ['logR']
to_T = lambda df, q: np.asarray(q) * df.dose.values / df.conc.values     # qe -> removal-proportional target
from_T = lambda df, t: np.asarray(t) * df.conc.values / df.dose.values   # exact inverse
GRID2 = [(h, a) for h in (4, 8, 12) for a in (1e-2, 1e-1)]
ENRICHED = MATS[1:]

def by_mat(tr, te, f, mats=MATS):
    out = pd.Series(np.nan, index=te.index)
    for m in mats:
        a, b = tr[tr.mat == m], te[te.mat == m]
        if len(b): out[b.index] = f(a, b)
    return out.values

def rsm_weighted(a, b):
    v = a.groupby('cond').y.var()
    v = v.clip(lower=v.median())          # floor so no condition gets extreme weight
    return rsm_predict(fit_rsm(a, w=1 / a.cond.map(v).values), b)

def rsm_removal(a, b): return from_T(b, rsm_predict(fit_rsm(a.assign(y=to_T(a, a.y))), b))

def gpr(a, b, removal=False):
    kern = ConstantKernel(1.0) * RBF(np.ones(len(PHYS)), (1e-2, 1e2)) + WhiteKernel(0.1, (1e-5, 10))
    sc = StandardScaler().fit(a[PHYS])
    y = to_T(a, a.y) if removal else a.y.values
    g = GaussianProcessRegressor(kern, normalize_y=True, n_restarts_optimizer=2, random_state=SEED).fit(sc.transform(a[PHYS]), y)
    p = g.predict(sc.transform(b[PHYS]))
    return from_T(b, p) if removal else p

def hgb(tr, te, mono):
    X = lambda df: np.column_stack([df[PHYS].values, df.mat.map({m: i for i, m in enumerate(MATS)}).values])
    Xa, ya = X(tr), tr.y.values
    cst = [1, 0, 0, -1, 1, 0] if mono else None   # conc up, dose down, conc/dose up
    mk = lambda lr, leaves: HistGradientBoostingRegressor(learning_rate=lr, max_leaf_nodes=leaves, max_iter=300, min_samples_leaf=5,
                                                          monotonic_cst=cst, categorical_features=[5], random_state=SEED)
    folds = list(GroupKFold(4).split(Xa, ya, tr.cond.values))
    cfgs = [(lr, leaves) for lr in (0.05, 0.1) for leaves in (7, 15)]
    best = min(cfgs, key=lambda c: np.mean([mean_squared_error(ya[v], mk(*c).fit(Xa[t], ya[t]).predict(Xa[v])) for t, v in folds]))
    return mk(*best).fit(Xa, ya).predict(X(te))

def mono_boost(tr, te):
    out = pd.Series(np.nan, index=te.index)
    e_tr, e_te = tr[tr.mat.isin(ENRICHED)], te[te.mat.isin(ENRICHED)]
    out[e_te.index] = hgb(e_tr, e_te, True)
    b_tr, b_te = tr[~tr.mat.isin(ENRICHED)], te[~te.mat.isin(ENRICHED)]
    if len(b_te): out[b_te.index] = hgb(b_tr, b_te, False)   # control: no constraint
    return out.values

def hybrid(tr, te):
    base = lambda a, b: rsm_predict(fit_rsm(a), b)
    resid = tr.y.values - by_mat(tr, tr, base)
    return by_mat(tr, te, base) + ann_predict(fit_ann(tr, 10, PHYS, target=resid, grid=GRID2, k=3), te)

CANDIDATES = {
    'RSM weighted': lambda tr, te: by_mat(tr, te, rsm_weighted),
    'RSM removal-space': lambda tr, te: by_mat(tr, te, rsm_removal),
    'ANN + conc/dose': lambda tr, te: ann_predict(fit_ann(tr, 10, PHYS, grid=GRID2, k=3), te),
    'ANN removal-space': lambda tr, te: from_T(te, ann_predict(fit_ann(tr, 10, PHYS, target=to_T(tr, tr.y), grid=GRID2, k=3), te)),
    'GPR + conc/dose': lambda tr, te: by_mat(tr, te, gpr),
    'GPR removal-space': lambda tr, te: by_mat(tr, te, lambda a, b: gpr(a, b, True)),
    'Monotonic boosting': mono_boost,
    'RSM + ANN residual': hybrid,
}
NEW = list(CANDIDATES) + ['Ensemble (ANN+GPR)']
ALLM = MODELS + NEW

def run_candidates(tr, te):
    warnings.filterwarnings('ignore')
    out = te.copy()
    for k, f in CANDIDATES.items(): out[k] = f(tr, te)
    out['Ensemble (ANN+GPR)'] = out[['ANN + conc/dose', 'GPR + conc/dose']].mean(axis=1)
    return out

hold2 = run_candidates(train, test).join(holdout[MODELS])
print('hold-out done')
res = Parallel(n_jobs=-1, verbose=5)(delayed(run_candidates)(d[d.cond != c], d[d.cond == c]) for c in conds)
loco2 = pd.concat(res).join(loco[MODELS])
assert d.y.equals(Y_ORIG), 'measured values changed!'
print('Check passed: all 480 measured qe values are unchanged.')

In [ ]:
t2, l2 = score(hold2, ALLM), score(loco2, ALLM)
comparison = (t2[t2.Material == 'ALL'].drop(columns='Material').set_index('Model').add_prefix('Test ')
              .join(l2[l2.Material == 'ALL'].drop(columns='Material').set_index('Model').add_prefix('LOCO '))
              .sort_values('LOCO RMSE'))
BEST = comparison.index[0]
print('Best model (lowest leave-one-out error):', BEST)
display(comparison.round(3))
per_material = (t2[t2.Model.isin(['RSM reduced', 'ANN', BEST])].pivot(index='Material', columns='Model', values='R2').add_prefix('Test R2 ')
                .join(l2[l2.Model.isin(['RSM reduced', 'ANN', BEST])].pivot(index='Material', columns='Model', values='R2').add_prefix('LOCO R2 ')))
display(per_material.round(3))

# Add the best model's unseen (LOCO) prediction to the actual-vs-predicted table, and its prediction at the lab validation points
avp = avp.merge(loco2.groupby(['mat', 'run'])[BEST].mean().rename(f'{BEST} unseen (LOCO)').reset_index(), on=['mat', 'run'])
vp = validation_plan.rename(columns={'Material': 'mat'}).copy()
for k in X4: vp[k] = to_coded(k, vp[LEVELS[k][0]])
vp['conc'], vp['dose'] = vp['Concentration (ppm)'], vp['Dosage (mg)']
vp['logR'] = np.log(vp.conc / vp.dose); vp['cond'] = -1
best_val = run_candidates(d, vp)[NEW] if BEST in NEW else None
if best_val is not None: validation_plan[f'{BEST} qe pred'] = best_val[BEST].values.round(2)
validation_plan

## Export everything

In [ ]:
with pd.ExcelWriter(f'{OUT}/RSM_ANN_results.xlsx') as xw:
    fit_summary.to_excel(xw, sheet_name='Fit summary', index=False)
    for i, m in enumerate(MATS): anova[m].to_excel(xw, sheet_name=f'ANOVA {i + 1} {m[:20]}'.replace(':', '-'), index=False)
    optimum.to_excel(xw, sheet_name='Optimum pH 4-8', index=False)
    optimum_ph4_12.to_excel(xw, sheet_name='Optimum pH 4-12 (compare)', index=False)
    validation_plan.to_excel(xw, sheet_name='Lab validation plan', index=False)
    ranking.to_excel(xw, sheet_name='Material ranking')
    two_way.to_excel(xw, sheet_name='Two-way ANOVA')
    tukey_df.to_excel(xw, sheet_name='Tukey HSD', index=False)
    avp.to_excel(xw, sheet_name='Actual vs predicted', index=False)
    test_metrics.to_excel(xw, sheet_name='Hold-out test metrics', index=False)
    loco_metrics.to_excel(xw, sheet_name='LOCO metrics', index=False)
    pd.DataFrame({'test runs (unseen)': test_conds}).to_excel(xw, sheet_name='Split', index=False)
    comparison.to_excel(xw, sheet_name='Model comparison')
    per_material.to_excel(xw, sheet_name='Best model per material')
    t2.to_excel(xw, sheet_name='All models test metrics', index=False)
    l2.to_excel(xw, sheet_name='All models LOCO metrics', index=False)
with zipfile.ZipFile('RSM_ANN_results.zip', 'w') as z:
    for f in os.listdir(OUT): z.write(f'{OUT}/{f}')
print(sorted(os.listdir(OUT)))
try:
    from google.colab import files; files.download('RSM_ANN_results.zip')
except ImportError:
    pass